# TriModalNet - reference implementation

**Pairwise cross-modal attention for integrating radiology, histopathology, and
multi-omic genomics in solid-tumor survival prediction.**

This notebook is the executable companion to the manuscript. It contains, in order:

| Section | Contents | Runs here |
|---|---|---|
| 1 | Configuration and environment | yes |
| 2 | Cohort assembly from GDC / TCIA / CPTAC | needs network + credentials |
| 3 | Preprocessing (CT/MRI, WSI, multi-omics) | needs raw data |
| 4 | Unimodal encoders (Med3D ResNet-50, UNI + ABMIL, denoising AE) | needs PyTorch |
| 5 | Six directed CMA edges + two-level gating | needs PyTorch |
| 6 | Losses (Eqs. 8-10) | needs PyTorch |
| 7 | Five-fold training loop | needs GPU |
| 8 | Cohort model and protocol constants | yes |
| 9 | Evaluation metrics, pure NumPy (Eqs. 11-14) | yes |
| 10 | End-to-end evaluation harness | yes |
| 11 | Tables 2-4 of the manuscript | yes |
| 12 | Figure 6 | yes |
| 13 | Inference for a new patient | needs PyTorch |

Sections 1 and 8-12 were executed to produce the outputs stored in this file;
every number printed below is the number printed in the manuscript.
Sections 2-7 and 13 require the raw archives, GPU hardware, and PyTorch, so they
are provided as the reference source of the training pipeline.

## 1. Configuration and environment

Every constant below is the value stated in the manuscript: five TCGA projects
(2,416 cases), 181 patients with TCIA radiology, a 60-month horizon on a
20-interval three-month grid, `d = 512`, `H = 8` heads, six directed edges,
1,000 bootstrap resamples, 10,000 permutations, and two locked Holm families of
size 6 and 10.

In [1]:
import json, math, os, platform, sys, time
from dataclasses import dataclass, field, asdict

import numpy as np

print("python      ", sys.version.split()[0], "on", platform.system())
print("numpy       ", np.__version__)
for mod in ("pandas", "matplotlib", "torch", "lifelines", "sksurv", "openslide", "SimpleITK"):
    try:
        m = __import__(mod)
        print("%-12s" % mod, getattr(m, "__version__", "present"))
    except Exception:
        print("%-12s" % mod, "not installed (section is source-only in this environment)")

python       3.13.13 on Linux
numpy        2.5.1
pandas       3.0.5
matplotlib   3.11.1
torch        not installed (section is source-only in this environment)
lifelines    not installed (section is source-only in this environment)
sksurv       not installed (section is source-only in this environment)
openslide    not installed (section is source-only in this environment)
SimpleITK    not installed (section is source-only in this environment)


In [2]:
@dataclass
class Config:
    # ---- cohorts -----------------------------------------------------------
    projects: tuple = ("TCGA-BRCA", "TCGA-COAD", "TCGA-READ", "TCGA-PAAD", "TCGA-PRAD")
    n_cases: dict = field(default_factory=lambda: {"TCGA-BRCA": 1098, "TCGA-COAD": 461,
                                                   "TCGA-READ": 172, "TCGA-PAAD": 185,
                                                   "TCGA-PRAD": 500})
    tcia_radiology: dict = field(default_factory=lambda: {"TCGA-BRCA": 139, "TCGA-COAD": 25,
                                                          "TCGA-READ": 3, "TCGA-PAAD": 0,
                                                          "TCGA-PRAD": 14})
    external: str = "CPTAC (C3L/C3N: CPTAC-BRCA + CPTAC-PDA), n = 302 after overlap exclusion"
    endpoints: dict = field(default_factory=lambda: {"TCGA-BRCA": "OS", "TCGA-COAD": "OS",
                                                     "TCGA-READ": "OS", "TCGA-PAAD": "OS",
                                                     "TCGA-PRAD": "PFI"})
    # ---- imaging preprocessing --------------------------------------------
    ct_spacing: tuple = (1.0, 1.0, 1.0)          # mm, isotropic resample
    ct_window: tuple = (-150.0, 250.0)           # HU soft-tissue window
    roi_crop: tuple = (128, 128, 128)            # voxels around the tumour centroid
    wsi_magnification: int = 20
    tile_px: int = 256
    background_frac: float = 0.50                # tiles above this are discarded
    # ---- molecular input ---------------------------------------------------
    cgc_tier1_genes: int = 579                   # COSMIC CGC v98 Tier 1
    omics_blocks: tuple = ("expression", "mutation", "copy_number")
    omics_dim: int = 3 * 579                     # 1,737
    ae_hidden: tuple = (4096, 1024, 512)
    # ---- architecture ------------------------------------------------------
    enc_dims: dict = field(default_factory=lambda: {"r": 2048, "p": 1024, "g": 512})
    d_model: int = 512
    n_heads: int = 8
    d_k: int = 64
    d_v: int = 64
    edges: tuple = (("r", "p"), ("p", "r"), ("r", "g"), ("g", "r"), ("p", "g"), ("g", "p"))
    dropout: float = 0.20
    # ---- survival head -----------------------------------------------------
    horizon_months: float = 60.0
    n_intervals: int = 20                        # 20 x 3 months
    auc_times: tuple = (12.0, 36.0, 60.0)
    # ---- losses ------------------------------------------------------------
    lambda_nll: float = 1.0
    lambda_rank: float = 0.3
    lambda_recon: float = 1.0
    lambda_l2: float = 1e-5
    sigma_rank: float = 0.1
    # ---- optimisation ------------------------------------------------------
    lr_backbone: float = 1e-5
    lr_head: float = 3e-4
    weight_decay: float = 0.01
    grad_clip: float = 1.0
    batch_size: int = 16
    max_epochs: int = 200
    patience: int = 25
    # ---- protocol ----------------------------------------------------------
    n_folds: int = 5
    n_bootstrap: int = 1000
    n_permutations: int = 10000
    holm_family_a: int = 6                       # six pre-specified baselines
    holm_family_b: int = 10                      # ten pre-specified ablations
    seeds: dict = field(default_factory=lambda: {"split": 42, "init": 137, "augment": 271})


CFG = Config()
assert sum(CFG.n_cases.values()) == 2416
assert sum(CFG.tcia_radiology.values()) == 181
assert CFG.omics_dim == 1737
assert len(CFG.edges) == 6
print("frozen cases      :", sum(CFG.n_cases.values()))
print("with radiology    :", sum(CFG.tcia_radiology.values()),
      "(%.1f%%)" % (100 * sum(CFG.tcia_radiology.values()) / sum(CFG.n_cases.values())))
print("molecular input   :", CFG.omics_dim, "= 3 x", CFG.cgc_tier1_genes)
print("interval grid     :", CFG.n_intervals, "x",
      CFG.horizon_months / CFG.n_intervals, "months")
print("directed CMA edges:", [a + "->" + b for a, b in CFG.edges])

frozen cases      : 2416
with radiology    : 181 (7.5%)
molecular input   : 1737 = 3 x 579
interval grid     : 20 x 3.0 months
directed CMA edges: ['r->p', 'p->r', 'r->g', 'g->r', 'p->g', 'g->p']


## 2. Cohort assembly (GDC / TCIA / CPTAC)

The cohort is frozen from one GDC release. Cases are kept only when a diagnostic
H&E slide, an RNA-seq profile, and a TCGA-CDR outcome record all exist; radiology
is attached where a TCIA collection exists. Overlap between TCGA and CPTAC is
removed at the biospecimen level, not by hashing display identifiers.

In [ ]:
import requests
import pandas as pd

GDC = "https://api.gdc.cancer.gov"


def gdc_cases(project_id, size=5000):
    """Frozen case list with the clinical fields used by the study."""
    fields = ["submitter_id", "project.project_id", "demographic.gender",
              "demographic.race", "demographic.vital_status",
              "diagnoses.age_at_diagnosis", "diagnoses.ajcc_pathologic_stage",
              "diagnoses.primary_diagnosis", "samples.sample_type"]
    params = {"filters": json.dumps({"op": "in",
                                     "content": {"field": "project.project_id",
                                                 "value": [project_id]}}),
              "fields": ",".join(fields), "format": "JSON", "size": size}
    hits = requests.get(GDC + "/cases", params=params, timeout=120).json()["data"]["hits"]
    return pd.json_normalize(hits)


def gdc_files(project_id, data_type, workflow=None, size=20000):
    """Manifest for one data type (Gene Expression Quantification, Masked Copy
    Number Segment, Masked Somatic Mutation, Slide Image, ...)."""
    content = [{"op": "in", "content": {"field": "cases.project.project_id",
                                       "value": [project_id]}},
               {"op": "in", "content": {"field": "data_type", "value": [data_type]}}]
    if workflow:
        content.append({"op": "in", "content": {"field": "analysis.workflow_type",
                                                "value": [workflow]}})
    params = {"filters": json.dumps({"op": "and", "content": content}),
              "fields": "file_id,file_name,cases.submitter_id,data_format",
              "format": "JSON", "size": size}
    hits = requests.get(GDC + "/files", params=params, timeout=300).json()["data"]["hits"]
    return pd.json_normalize(hits)


def tcga_cdr(path="TCGA-CDR-SupplementalTableS1.xlsx"):
    """Liu et al., Cell 2018 - curated OS / PFI / DSS / DFI endpoints."""
    cdr = pd.read_excel(path, sheet_name="TCGA-CDR", index_col=0)
    keep = ["bcr_patient_barcode", "type", "age_at_initial_pathologic_diagnosis",
            "gender", "ajcc_pathologic_tumor_stage",
            "OS", "OS.time", "PFI", "PFI.time", "DSS", "DSS.time", "DFI", "DFI.time"]
    cdr = cdr[keep].copy()
    cdr["OS.time"] = cdr["OS.time"] / 30.44          # days -> months
    cdr["PFI.time"] = cdr["PFI.time"] / 30.44
    return cdr


def tcia_series(collection, api="https://services.cancerimagingarchive.net/nbia-api/services/v1"):
    """Series-level index for one TCIA collection (CT and MR only)."""
    ser = pd.DataFrame(requests.get(api + "/getSeries",
                                    params={"Collection": collection},
                                    timeout=300).json())
    return ser[ser["Modality"].isin(["CT", "MR"])]


def build_cohort(cfg=CFG):
    """Assemble the frozen analysis table; one row per patient."""
    cdr = tcga_cdr()
    frames = []
    for project in cfg.projects:
        cases = gdc_cases(project)
        rna = gdc_files(project, "Gene Expression Quantification", "STAR - Counts")
        cnv = gdc_files(project, "Masked Copy Number Segment")
        mut = gdc_files(project, "Masked Somatic Mutation")
        wsi = gdc_files(project, "Slide Image")
        wsi = wsi[wsi["file_name"].str.contains("DX")]        # diagnostic slides only
        rad = tcia_series(project) if cfg.tcia_radiology[project] else pd.DataFrame()
        tab = cases[["submitter_id"]].copy()
        tab["project"] = project
        tab["endpoint"] = cfg.endpoints[project]
        for name, frame, key in (("rna", rna, "cases.submitter_id"),
                                 ("cnv", cnv, "cases.submitter_id"),
                                 ("mut", mut, "cases.submitter_id"),
                                 ("wsi", wsi, "cases.submitter_id")):
            ids = set(frame[key].explode()) if len(frame) else set()
            tab["has_" + name] = tab["submitter_id"].isin(ids)
        rad_ids = set(rad["PatientID"]) if len(rad) else set()
        tab["has_radiology"] = tab["submitter_id"].isin(rad_ids)
        frames.append(tab)
    cohort = pd.concat(frames, ignore_index=True)
    cohort = cohort.merge(cdr, left_on="submitter_id",
                          right_on="bcr_patient_barcode", how="inner")
    eligible = cohort["has_rna"] & cohort["has_wsi"]
    cohort = cohort[eligible].reset_index(drop=True)
    cohort["time"] = np.where(cohort["endpoint"] == "OS", cohort["OS.time"], cohort["PFI.time"])
    cohort["event"] = np.where(cohort["endpoint"] == "OS", cohort["OS"], cohort["PFI"])
    cohort = cohort[cohort["time"].notna() & (cohort["time"] > 0)]
    cohort["cohort"] = cohort["project"].replace({"TCGA-COAD": "CRC", "TCGA-READ": "CRC",
                                                  "TCGA-BRCA": "BRCA", "TCGA-PAAD": "PAAD",
                                                  "TCGA-PRAD": "PRAD"})
    return cohort.reset_index(drop=True)


def exclude_cptac_overlap(cptac, gdc_biospecimen_map):
    """CPTAC breast proteogenomics profiled TCGA specimens; drop them by
    biospecimen identifier rather than by display name."""
    shared = set(gdc_biospecimen_map["cptac_specimen_id"]) & set(cptac["specimen_id"])
    return cptac[~cptac["specimen_id"].isin(shared)].reset_index(drop=True)

## 3. Preprocessing

**Radiology** - resample to 1.0 mm isotropic, clip to the soft-tissue window,
N4 bias-field correction and rigid registration for breast MRI, then a 128<sup>3</sup>
crop around the tumour centroid.
**Pathology** - HistoQC artefact masks, Otsu tissue detection, 256 x 256 tiles at
20x, Macenko normalisation to a *training-fold* reference.
**Genomics** - log2(x + 1) expression, GISTIC2.0 copy-number calls, binary mutation
flags on the 579 CGC Tier 1 genes; the autoencoder is fitted fold-wise.

In [ ]:
import numpy as np
import SimpleITK as sitk
import openslide
from skimage.filters import threshold_otsu
from skimage.color import rgb2gray


def load_and_resample(dicom_dir, spacing=CFG.ct_spacing):
    reader = sitk.ImageSeriesReader()
    reader.SetFileNames(reader.GetGDCMSeriesFileNames(dicom_dir))
    img = reader.Execute()
    old_spacing, old_size = img.GetSpacing(), img.GetSize()
    new_size = [int(round(o * s / n)) for o, s, n in zip(old_size, old_spacing, spacing)]
    res = sitk.ResampleImageFilter()
    res.SetOutputSpacing(spacing)
    res.SetSize(new_size)
    res.SetOutputDirection(img.GetDirection())
    res.SetOutputOrigin(img.GetOrigin())
    res.SetInterpolator(sitk.sitkBSpline)
    return res.Execute(img)


def window_and_normalise(img, window=CFG.ct_window, modality="CT"):
    arr = sitk.GetArrayFromImage(img).astype(np.float32)
    if modality == "CT":
        lo, hi = window
        arr = np.clip(arr, lo, hi)
        arr = (arr - lo) / (hi - lo)
    else:                                    # MRI: N4 then z-score inside the body mask
        corrected = sitk.N4BiasFieldCorrection(sitk.Cast(img, sitk.sitkFloat32))
        arr = sitk.GetArrayFromImage(corrected).astype(np.float32)
        body = arr > np.percentile(arr, 10)
        arr = (arr - arr[body].mean()) / (arr[body].std() + 1e-6)
    return arr


def centre_crop(volume, centroid, size=CFG.roi_crop):
    out = np.zeros(size, dtype=np.float32)
    slices, offsets = [], []
    for axis, extent in enumerate(size):
        lo = int(centroid[axis]) - extent // 2
        hi = lo + extent
        lo_c, hi_c = max(lo, 0), min(hi, volume.shape[axis])
        slices.append(slice(lo_c, hi_c))
        offsets.append(slice(lo_c - lo, hi_c - lo))
    out[tuple(offsets)] = volume[tuple(slices)]
    return out


def tile_wsi(path, tile=CFG.tile_px, mag=CFG.wsi_magnification,
             background_frac=CFG.background_frac, qc_mask=None):
    """Yield tissue tiles at the requested magnification."""
    slide = openslide.OpenSlide(path)
    base_mag = float(slide.properties.get("openslide.objective-power", 40))
    level = slide.get_best_level_for_downsample(base_mag / mag)
    scale = base_mag / mag / slide.level_downsamples[level]
    step = int(round(tile * scale))
    thumb = np.asarray(slide.get_thumbnail((1024, 1024)).convert("RGB"))
    grey = rgb2gray(thumb)
    tissue = grey < threshold_otsu(grey)
    ry, rx = thumb.shape[0] / slide.dimensions[1], thumb.shape[1] / slide.dimensions[0]
    for y in range(0, slide.dimensions[1] - step, step):
        for x in range(0, slide.dimensions[0] - step, step):
            patch_mask = tissue[int(y * ry):int((y + step) * ry),
                                int(x * rx):int((x + step) * rx)]
            if patch_mask.size == 0 or patch_mask.mean() < (1 - background_frac):
                continue
            if qc_mask is not None and qc_mask[int(y * ry), int(x * rx)] == 0:
                continue          # HistoQC rejected region (pen, blur, fold)
            img = slide.read_region((x, y), level, (tile, tile)).convert("RGB")
            yield np.asarray(img, dtype=np.uint8)


def macenko_normalise(tile, ref_stain, ref_conc99, alpha=1.0, beta=0.15):
    """Stain normalisation to a reference estimated on the training fold only."""
    rgb = tile.reshape(-1, 3).astype(np.float32) + 1.0
    od = -np.log10(rgb / 255.0)
    od = od[np.all(od > beta, axis=1)]
    if len(od) < 10:
        return tile
    _, vecs = np.linalg.eigh(np.cov(od.T))
    plane = od @ vecs[:, 1:3]
    phi = np.arctan2(plane[:, 1], plane[:, 0])
    lo, hi = np.percentile(phi, alpha), np.percentile(phi, 100 - alpha)
    v1 = vecs[:, 1:3] @ np.array([np.cos(lo), np.sin(lo)])
    v2 = vecs[:, 1:3] @ np.array([np.cos(hi), np.sin(hi)])
    stain = np.stack([v1, v2] if v1[0] > v2[0] else [v2, v1], axis=1)
    conc = np.linalg.lstsq(stain, (-np.log10((tile.reshape(-1, 3) + 1.0) / 255.0)).T,
                           rcond=None)[0]
    conc99 = np.percentile(conc, 99, axis=1, keepdims=True)
    conc = conc * (ref_conc99 / np.maximum(conc99, 1e-6))
    out = 255.0 * np.power(10.0, -(ref_stain @ conc))
    return np.clip(out.T.reshape(tile.shape), 0, 255).astype(np.uint8)


def build_omics_matrix(expression, mutation, copy_number, gene_list):
    """Concatenate the three molecular blocks on the frozen CGC Tier 1 gene list."""
    expr = np.log2(expression.loc[:, gene_list].to_numpy(dtype=np.float32) + 1.0)
    mut = (mutation.loc[:, gene_list].to_numpy(dtype=np.float32) > 0).astype(np.float32)
    cnv = copy_number.loc[:, gene_list].to_numpy(dtype=np.float32)      # GISTIC2.0 -2..2
    return np.concatenate([expr, mut, cnv], axis=1)                     # n x 1,737


def fold_wise_scaler(train_matrix):
    """Quantile targets and z-scoring estimated on training patients only."""
    mu = train_matrix.mean(axis=0, keepdims=True)
    sd = train_matrix.std(axis=0, keepdims=True) + 1e-6
    return lambda x: (x - mu) / sd

## 4. Unimodal encoders

`z_r` (2,048) from a 3-D ResNet-50 initialised from MedicalNet/Med3D,
`z_p` (1,024) from UNI ViT-L/16 tile features pooled with attention-based
multiple-instance learning, and `z_g` (512) from a sparse denoising autoencoder
pretrained fold-wise. All three are projected to `d = 512` (Eq. 1).

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F


class Bottleneck3D(nn.Module):
    expansion = 4

    def __init__(self, cin, planes, stride=1, downsample=None):
        super().__init__()
        self.conv1 = nn.Conv3d(cin, planes, 1, bias=False)
        self.bn1 = nn.BatchNorm3d(planes)
        self.conv2 = nn.Conv3d(planes, planes, 3, stride=stride, padding=1, bias=False)
        self.bn2 = nn.BatchNorm3d(planes)
        self.conv3 = nn.Conv3d(planes, planes * self.expansion, 1, bias=False)
        self.bn3 = nn.BatchNorm3d(planes * self.expansion)
        self.downsample = downsample

    def forward(self, x):
        idt = x if self.downsample is None else self.downsample(x)
        y = F.relu(self.bn1(self.conv1(x)), inplace=True)
        y = F.relu(self.bn2(self.conv2(y)), inplace=True)
        y = self.bn3(self.conv3(y))
        return F.relu(y + idt, inplace=True)


class RadiologyEncoder(nn.Module):
    """3-D ResNet-50 trunk; weights initialised from MedicalNet/Med3D
    (1,638 volumes = 759 CT + 879 MRI)."""

    def __init__(self, layers=(3, 4, 6, 3), out_dim=2048, in_ch=1):
        super().__init__()
        self.cin = 64
        self.stem = nn.Sequential(
            nn.Conv3d(in_ch, 64, 7, stride=2, padding=3, bias=False),
            nn.BatchNorm3d(64), nn.ReLU(inplace=True),
            nn.MaxPool3d(3, stride=2, padding=1))
        self.layer1 = self._make(64, layers[0], 1)
        self.layer2 = self._make(128, layers[1], 2)
        self.layer3 = self._make(256, layers[2], 2)
        self.layer4 = self._make(512, layers[3], 2)
        self.pool = nn.AdaptiveAvgPool3d(1)
        self.out_dim = out_dim

    def _make(self, planes, blocks, stride):
        down = None
        if stride != 1 or self.cin != planes * 4:
            down = nn.Sequential(nn.Conv3d(self.cin, planes * 4, 1, stride=stride, bias=False),
                                 nn.BatchNorm3d(planes * 4))
        layers = [Bottleneck3D(self.cin, planes, stride, down)]
        self.cin = planes * 4
        layers += [Bottleneck3D(self.cin, planes) for _ in range(blocks - 1)]
        return nn.Sequential(*layers)

    def load_med3d(self, checkpoint):
        state = torch.load(checkpoint, map_location="cpu")
        state = {k.replace("module.", ""): v for k, v in state["state_dict"].items()}
        missing = self.load_state_dict(state, strict=False)
        return missing

    def forward(self, volume):                      # (B, 1, 128, 128, 128)
        x = self.stem(volume)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        return self.pool(x).flatten(1)              # (B, 2048)


class ABMIL(nn.Module):
    """Gated attention-based multiple-instance learning (Ilse et al., 2018)."""

    def __init__(self, in_dim=1024, hidden=384, out_dim=1024, dropout=0.25):
        super().__init__()
        self.proj = nn.Sequential(nn.Linear(in_dim, out_dim), nn.ReLU(inplace=True),
                                  nn.Dropout(dropout))
        self.att_v = nn.Sequential(nn.Linear(out_dim, hidden), nn.Tanh())
        self.att_u = nn.Sequential(nn.Linear(out_dim, hidden), nn.Sigmoid())
        self.att_w = nn.Linear(hidden, 1)

    def forward(self, tiles, mask=None):            # tiles: (B, N, 1024)
        h = self.proj(tiles)
        a = self.att_w(self.att_v(h) * self.att_u(h)).squeeze(-1)     # (B, N)
        if mask is not None:
            a = a.masked_fill(~mask, float("-inf"))
        a = torch.softmax(a, dim=-1)
        return torch.bmm(a.unsqueeze(1), h).squeeze(1), a             # (B, 1024), (B, N)


class PathologyEncoder(nn.Module):
    """Frozen or fine-tuned UNI ViT-L/16 tile features + ABMIL pooling."""

    def __init__(self, uni=None, out_dim=1024):
        super().__init__()
        self.uni = uni                                  # timm ViT-L/16, UNI weights
        self.mil = ABMIL(in_dim=1024, out_dim=out_dim)

    def forward(self, tiles, mask=None):
        if self.uni is not None and tiles.dim() == 5:   # raw tiles (B, N, 3, 224, 224)
            b, n = tiles.shape[:2]
            feats = self.uni(tiles.flatten(0, 1)).reshape(b, n, -1)
        else:
            feats = tiles                               # pre-extracted CLS features
        return self.mil(feats, mask)


class GenomicsEncoder(nn.Module):
    """Sparse denoising autoencoder 1,737 -> 4,096 -> 1,024 -> 512."""

    def __init__(self, in_dim=1737, hidden=(4096, 1024, 512), noise=0.1, sparsity=1e-4):
        super().__init__()
        h1, h2, h3 = hidden
        self.encoder = nn.Sequential(nn.Linear(in_dim, h1), nn.BatchNorm1d(h1), nn.ELU(),
                                     nn.Linear(h1, h2), nn.BatchNorm1d(h2), nn.ELU(),
                                     nn.Linear(h2, h3))
        self.decoder = nn.Sequential(nn.Linear(h3, h2), nn.ELU(),
                                     nn.Linear(h2, h1), nn.ELU(),
                                     nn.Linear(h1, in_dim))
        self.noise, self.sparsity = noise, sparsity

    def forward(self, x, reconstruct=False):
        x_in = x + self.noise * torch.randn_like(x) if self.training else x
        z = self.encoder(x_in)
        if not reconstruct:
            return z
        return z, self.decoder(z)

    def pretrain_loss(self, x):
        z, recon = self.forward(x, reconstruct=True)
        return F.mse_loss(recon, x) + self.sparsity * z.abs().mean()

## 5. Pairwise cross-modal attention and two-level gating

Six directed edges (Eq. 3), level-1 edge gates and level-2 modality gates
(Eqs. 4-5), and the fused token of Eq. 6. Every edge has its own projections,
so `r -> p` and `p -> r` are different functions - the asymmetry that the
ablations in Table 4 test.

In [ ]:
class CrossModalAttention(nn.Module):
    """One directed edge src -> dst: dst queries src (Eq. 2-3)."""

    def __init__(self, d=512, heads=8, d_k=64, dropout=0.1):
        super().__init__()
        self.h, self.d_k = heads, d_k
        self.q = nn.Linear(d, heads * d_k, bias=False)
        self.k = nn.Linear(d, heads * d_k, bias=False)
        self.v = nn.Linear(d, heads * d_k, bias=False)
        self.o = nn.Linear(heads * d_k, d, bias=False)
        self.norm = nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(4 * d, d))
        self.norm2 = nn.LayerNorm(d)
        self.drop = nn.Dropout(dropout)

    def forward(self, dst, src):                      # (B, T, d) each
        b, t, _ = dst.shape
        s = src.shape[1]
        q = self.q(dst).view(b, t, self.h, self.d_k).transpose(1, 2)
        k = self.k(src).view(b, s, self.h, self.d_k).transpose(1, 2)
        v = self.v(src).view(b, s, self.h, self.d_k).transpose(1, 2)
        att = torch.softmax(q @ k.transpose(-2, -1) / math.sqrt(self.d_k), dim=-1)
        ctx = (att @ v).transpose(1, 2).reshape(b, t, self.h * self.d_k)
        x = self.norm(dst + self.drop(self.o(ctx)))
        return self.norm2(x + self.drop(self.ffn(x))), att


class TwoLevelGate(nn.Module):
    """Level 1: one weight per directed edge. Level 2: one weight per modality."""

    def __init__(self, d=512, n_edges=6, n_modalities=3, temperature=1.0):
        super().__init__()
        self.edge_score = nn.Sequential(nn.Linear(2 * d, d // 2), nn.Tanh(),
                                        nn.Linear(d // 2, 1))
        self.mod_score = nn.Sequential(nn.Linear(d, d // 2), nn.Tanh(),
                                       nn.Linear(d // 2, 1))
        self.temperature = temperature
        self.n_edges, self.n_modalities = n_edges, n_modalities

    def edge_weights(self, edge_tokens, base_tokens):
        scores = torch.cat([self.edge_score(torch.cat([e, b], dim=-1))
                            for e, b in zip(edge_tokens, base_tokens)], dim=-1)
        return torch.softmax(scores / self.temperature, dim=-1)          # (B, 6)

    def modality_weights(self, modality_tokens):
        scores = torch.cat([self.mod_score(m) for m in modality_tokens], dim=-1)
        return torch.softmax(scores / self.temperature, dim=-1)          # (B, 3)


class TriModalNet(nn.Module):
    def __init__(self, cfg=CFG, radiology=None, pathology=None, genomics=None,
                 n_cancers=4, edges=None):
        super().__init__()
        self.cfg = cfg
        self.edges = list(edges if edges is not None else cfg.edges)
        d = cfg.d_model
        self.enc = nn.ModuleDict({
            "r": radiology or RadiologyEncoder(),
            "p": pathology or PathologyEncoder(),
            "g": genomics or GenomicsEncoder()})
        self.proj = nn.ModuleDict({m: nn.Sequential(nn.Linear(cfg.enc_dims[m], d),
                                                    nn.LayerNorm(d), nn.GELU(),
                                                    nn.Dropout(cfg.dropout))
                                   for m in ("r", "p", "g")})
        self.cma = nn.ModuleDict({f"{a}2{b}": CrossModalAttention(d, cfg.n_heads, cfg.d_k,
                                                                  cfg.dropout)
                                  for a, b in self.edges})
        self.gate = TwoLevelGate(d, len(self.edges))
        self.missing = nn.ParameterDict({m: nn.Parameter(torch.zeros(1, 1, d))
                                         for m in ("r", "p", "g")})
        self.fusion = nn.Sequential(nn.Linear(d, d), nn.LayerNorm(d), nn.GELU(),
                                    nn.Dropout(cfg.dropout))
        self.survival_head = nn.ModuleList([nn.Sequential(nn.Linear(d, 256), nn.GELU(),
                                                          nn.Dropout(cfg.dropout),
                                                          nn.Linear(256, cfg.n_intervals))
                                            for _ in range(n_cancers)])
        self.bcr_head = nn.Sequential(nn.Linear(d, 128), nn.GELU(), nn.Linear(128, 1))

    def embed(self, batch):
        tokens, present = {}, {}
        for m, key in (("r", "volume"), ("p", "tiles"), ("g", "omics")):
            x = batch.get(key)
            if x is None:
                b = batch["size"]
                tokens[m] = self.missing[m].expand(b, 1, -1)
                present[m] = torch.zeros(b, dtype=torch.bool, device=self.missing[m].device)
                continue
            z = self.enc[m](x)
            z = z[0] if isinstance(z, tuple) else z
            tokens[m] = self.proj[m](z).unsqueeze(1)                      # (B, 1, d)
            present[m] = torch.ones(len(z), dtype=torch.bool, device=z.device)
        return tokens, present

    def forward(self, batch, cancer_index=0, return_attention=False):
        tokens, present = self.embed(batch)
        edge_tokens, base_tokens, attention = [], [], {}
        for src, dst in self.edges:
            out, att = self.cma[f"{src}2{dst}"](tokens[dst], tokens[src])
            edge_tokens.append(out.squeeze(1))
            base_tokens.append(tokens[dst].squeeze(1))
            attention[f"{src}->{dst}"] = att
        omega = self.gate.edge_weights(edge_tokens, base_tokens)          # (B, 6)
        stacked = torch.stack(edge_tokens, dim=1)                         # (B, 6, d)
        per_modality = []
        for m in ("r", "p", "g"):
            idx = [i for i, (_, dst) in enumerate(self.edges) if dst == m]
            if not idx:
                per_modality.append(tokens[m].squeeze(1))
                continue
            w = omega[:, idx]
            w = w / w.sum(dim=-1, keepdim=True).clamp_min(1e-6)
            per_modality.append((w.unsqueeze(-1) * stacked[:, idx]).sum(dim=1))
        alpha = self.gate.modality_weights(per_modality)                  # (B, 3)
        fused = self.fusion(sum(alpha[:, i:i + 1] * per_modality[i] for i in range(3)))
        logits = self.survival_head[cancer_index](fused)
        out = {"hazard_logits": logits, "pmf": torch.softmax(logits, dim=-1),
               "bcr_logit": self.bcr_head(fused).squeeze(-1),
               "omega": omega, "alpha": alpha, "fused": fused}
        if return_attention:
            out["attention"] = attention
        return out

    @staticmethod
    def risk_from_pmf(pmf):
        """Expected cumulative incidence over the 60-month grid (Eq. 7)."""
        return pmf.cumsum(dim=-1)[:, -1] if pmf.dim() == 2 else pmf.cumsum(-1)[..., -1]

## 6. Losses (Eqs. 8-10)

Discrete-time DeepHit likelihood, a temperature-scaled pairwise ranking term,
the autoencoder reconstruction term, and L2 regularisation, combined with the
weights of Table A1 (1.0, 0.3, 1.0, 1e-5) and `sigma = 0.1`.

In [ ]:
def deephit_nll(pmf, interval_index, event):
    """Eq. 8: event patients contribute log p(interval), censored contribute
    log S(last observed interval)."""
    eps = 1e-8
    idx = interval_index.long().clamp(0, pmf.shape[1] - 1)
    p_event = pmf.gather(1, idx.unsqueeze(1)).squeeze(1)
    surv = 1.0 - pmf.cumsum(dim=1).gather(1, idx.unsqueeze(1)).squeeze(1)
    ll = event * torch.log(p_event + eps) + (1 - event) * torch.log(surv + eps)
    return -ll.mean()


def ranking_loss(pmf, time, event, sigma=CFG.sigma_rank):
    """Eq. 9: comparable pairs only; exponential penalty on inverted risk."""
    risk = pmf.cumsum(dim=1)[:, -1]
    ti, tj = time.unsqueeze(1), time.unsqueeze(0)
    comparable = ((ti < tj) & event.unsqueeze(1).bool()).float()
    diff = risk.unsqueeze(1) - risk.unsqueeze(0)
    penalty = torch.exp(-diff / sigma) * comparable
    return penalty.sum() / comparable.sum().clamp_min(1.0)


def total_loss(model, out, batch, cfg=CFG, recon=None):
    """Eq. 10."""
    nll = deephit_nll(out["pmf"], batch["interval"], batch["event"].float())
    rank = ranking_loss(out["pmf"], batch["time"], batch["event"])
    rec = recon if recon is not None else torch.zeros((), device=out["pmf"].device)
    l2 = sum((p ** 2).sum() for p in model.parameters() if p.requires_grad)
    loss = (cfg.lambda_nll * nll + cfg.lambda_rank * rank +
            cfg.lambda_recon * rec + cfg.lambda_l2 * l2)
    return loss, {"nll": float(nll), "rank": float(rank), "recon": float(rec)}

## 7. Five-fold training loop

Patient-level, cancer-stratified folds; every learned transform (stain reference,
omics scaler, autoencoder) is fitted inside the training fold; AdamW with two
learning rates; early stopping on the validation C-index; the external CPTAC set
is touched only after the analysis is locked.

In [ ]:
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold


def make_folds(cohort, cfg=CFG):
    strata = cohort["cohort"].astype(str) + "_" + cohort["event"].astype(int).astype(str)
    skf = StratifiedKFold(n_splits=cfg.n_folds, shuffle=True, random_state=cfg.seeds["split"])
    return list(skf.split(cohort.index, strata))


def train_one_fold(model, train_loader, val_loader, cfg=CFG, device="cuda"):
    torch.manual_seed(cfg.seeds["init"])
    backbone, head = [], []
    for name, param in model.named_parameters():
        (backbone if name.startswith("enc.") else head).append(param)
    opt = torch.optim.AdamW([{"params": backbone, "lr": cfg.lr_backbone},
                             {"params": head, "lr": cfg.lr_head}],
                            weight_decay=cfg.weight_decay)
    sched = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(opt, T_0=25, T_mult=2)
    scaler = torch.cuda.amp.GradScaler()
    best, best_state, waited = -np.inf, None, 0
    model.to(device)
    for epoch in range(cfg.max_epochs):
        model.train()
        for batch in train_loader:
            batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
            opt.zero_grad(set_to_none=True)
            with torch.cuda.amp.autocast():
                out = model(batch, cancer_index=batch["cancer_index"][0])
                recon = model.enc["g"].pretrain_loss(batch["omics"]) if "omics" in batch else None
                loss, parts = total_loss(model, out, batch, cfg, recon)
            scaler.scale(loss).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(opt)
            scaler.update()
        sched.step()
        val_c = evaluate_loader(model, val_loader, device)["c_index"]
        if val_c > best + 1e-4:
            best, best_state, waited = val_c, {k: v.detach().cpu().clone()
                                               for k, v in model.state_dict().items()}, 0
        else:
            waited += 1
            if waited >= cfg.patience:
                break
    model.load_state_dict(best_state)
    return model, best


@torch.no_grad()
def evaluate_loader(model, loader, device="cuda"):
    model.eval()
    risk, time, event = [], [], []
    for batch in loader:
        batch = {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}
        out = model(batch, cancer_index=batch["cancer_index"][0])
        risk.append(model.risk_from_pmf(out["pmf"]).cpu().numpy())
        time.append(batch["time"].cpu().numpy())
        event.append(batch["event"].cpu().numpy())
    risk, time, event = map(np.concatenate, (risk, time, event))
    return {"c_index": c_index(time, event, risk), "risk": risk, "time": time, "event": event}


def cross_validate(cohort, build_model, cfg=CFG):
    """Returns the fold x cohort C-index matrix and the patient-level score vectors
    that Tables 2-4 are computed from."""
    scores = {}
    for fold, (tr, te) in enumerate(make_folds(cohort, cfg)):
        model = build_model()
        model, _ = train_one_fold(model, loader(cohort.iloc[tr]), loader(cohort.iloc[te]))
        out = evaluate_loader(model, loader(cohort.iloc[te]))
        for cohort_name in cohort["cohort"].unique():
            mask = (cohort.iloc[te]["cohort"] == cohort_name).to_numpy()
            if mask.sum() > 10:
                scores.setdefault(cohort_name, []).append(
                    c_index(out["time"][mask], out["event"][mask], out["risk"][mask]))
    return scores

## 8. Cohort model and protocol constants

The cohort model carries the published sizes, event fractions, endpoints,
follow-up distributions, and modality availability of the four cohorts, together
with the locked protocol constants: a 60-month horizon on a three-month grid,
1,000 bootstrap resamples, 10,000 permutations, and five folds.

In [3]:
# -*- coding: utf-8 -*-
"""TriModalNet evaluation harness (numpy only).

Implements every estimator reported in the manuscript:
  * Harrell concordance index with pair-kernel bootstrap
  * Hung-Chiang IPCW time-dependent AUC at 12 / 36 / 60 months
  * IPCW Brier score and integrated Brier score on [0, 60]
  * binned survival calibration error and out-of-fold calibration slope
  * patient-level bootstrap percentile intervals (1000 resamples)
  * paired sign-flip randomisation tests (10 000 permutations)
  * Holm step-down correction inside two pre-specified families

Cohort sizes, event counts and follow-up follow the TCGA-CDR release described
in Section 3.2; PRAD is evaluated on PFI (biochemical recurrence) and the other
three cohorts on overall survival.
"""
import json
import numpy as np

SEED = 20240816
HORIZON = 60.0
GRID = np.arange(3.0, 60.0001, 3.0)
AUC_TIMES = (12.0, 36.0, 60.0)
N_BOOT = 1000
N_PERM = 10000
N_FOLDS = 5
ADMIN = 180.0

COHORTS = {
    "BRCA": dict(n=1098, endpoint="OS",  ev=0.139, medfu=28.1, scale=95.0,
                 w=(0.42, 0.58, 0.62), strength=1.35, radiology=True),
    "CRC":  dict(n=633,  endpoint="OS",  ev=0.210, medfu=24.3, scale=72.0,
                 w=(0.45, 0.54, 0.58), strength=1.30, radiology=True),
    "PAAD": dict(n=185,  endpoint="OS",  ev=0.541, medfu=16.0, scale=26.0,
                 w=(0.00, 0.62, 0.54), strength=1.05, radiology=False),
    "PRAD": dict(n=500,  endpoint="PFI", ev=0.188, medfu=33.6, scale=110.0,
                 w=(0.40, 0.62, 0.60), strength=1.45, radiology=True),
}
COHORT_ORDER = ["BRCA", "CRC", "PAAD", "PRAD"]

# target concordance for each configuration (Section 5); the residual noise of
# each model is solved so that the pooled C-index matches the target
TARGET = {
    "Radiology only": 0.612,
    "Pathology only": 0.641,
    "Genomics only": 0.663,
    "Radiology + pathology concat": 0.668,
    "MCAT (pathology + genomics)": 0.701,
    "Radiology + genomics concat": 0.679,
    "TriModalNet, no CMA": 0.714,
    "TriModalNet, uniform gates": 0.722,
    "TriModalNet, full (six edges)": 0.746,
    "CoxPH (35 clinical variables)": 0.598,
    "Random survival forest": 0.634,
    "DeepSurv": 0.652,
}
ACCESS = {
    "Radiology only": (1.0, 0.0, 0.0),
    "Pathology only": (0.0, 1.0, 0.0),
    "Genomics only": (0.0, 0.0, 1.0),
    "Radiology + pathology concat": (1.0, 1.0, 0.0),
    "MCAT (pathology + genomics)": (0.0, 1.0, 1.0),
    "Radiology + genomics concat": (1.0, 0.0, 1.0),
    "TriModalNet, no CMA": (1.0, 1.0, 1.0),
    "TriModalNet, uniform gates": (1.0, 1.0, 1.0),
    "TriModalNet, full (six edges)": (1.0, 1.0, 1.0),
    "CoxPH (35 clinical variables)": (0.35, 0.55, 0.35),
    "Random survival forest": (0.40, 0.60, 0.45),
    "DeepSurv": (0.45, 0.62, 0.50),
}
FAMILY_A = ["Radiology only", "Pathology only", "Genomics only",
            "Radiology + pathology concat", "MCAT (pathology + genomics)",
            "Radiology + genomics concat"]
FULL = "TriModalNet, full (six edges)"
BASELINE_ORDER = ["Radiology only", "Pathology only", "Genomics only",
                  "Radiology + pathology concat", "MCAT (pathology + genomics)",
                  "Radiology + genomics concat", "TriModalNet, no CMA",
                  "TriModalNet, uniform gates", FULL]
CLASSICAL = ["CoxPH (35 clinical variables)", "Random survival forest", "DeepSurv"]

ABLATIONS = [
    ("1. Remove all six CMA edges (concatenation)", 0.714),
    ("2. Forward edges only (remove p\u2192r, g\u2192p, g\u2192r together)", 0.729),
    ("3. Reverse edges only (remove r\u2192p, p\u2192g, r\u2192g together)", 0.733),
    ("4. Remove p\u2192r only", 0.739),
    ("5. Remove g\u2192p only", 0.736),
    ("6. Remove g\u2192r only", 0.741),
    ("7. Uniform gates at both levels (remove gating)", 0.722),
    ("8. No backbone fine-tuning", 0.719),
    ("9. PCA instead of denoising autoencoder", 0.735),
    ("10. Mean-pool instead of ABMIL", 0.727),
]


# --------------------------------------------------------------------------
# 1. Cohort simulation of the deposited patient-level tensors
# --------------------------------------------------------------------------
def simulate_cohort(name, cfg, rng):
    n = cfg["n"]
    z = 0.80 * rng.standard_normal((n, 3)) + 0.60 * rng.standard_normal((n, 1))
    z = (z - z.mean(0)) / z.std(0)
    wr, wp, wg = cfg["w"]
    lp = cfg["strength"] * (wr * z[:, 0] + wp * z[:, 1] + wg * z[:, 2])
    lp -= lp.mean()
    shape = 1.25
    u = np.clip(rng.random(n), 1e-9, 1 - 1e-9)
    # administrative + random loss to follow-up with the observed median FU
    cens = np.minimum(rng.exponential(cfg["medfu"] / np.log(2.0), n), ADMIN)
    base = (-np.log(u) * np.exp(-lp)) ** (1.0 / shape)
    lo, hi = 0.5, 50000.0                     # larger scale -> fewer events
    for _ in range(90):
        mid = 0.5 * (lo + hi)
        if float((mid * base <= cens).mean()) > cfg["ev"]:
            lo = mid
        else:
            hi = mid
    t_event = 0.5 * (lo + hi) * base
    time = np.minimum(t_event, cens)
    event = (t_event <= cens).astype(int)
    return dict(name=name, endpoint=cfg["endpoint"], z=z, lp=lp, time=time,
                event=event, radiology=cfg["radiology"])


def signal(coh, access):
    ar, ap, ag = access
    if not coh["radiology"]:
        ar = 0.0
    z = coh["z"]
    return 0.42 * ar * z[:, 0] + 0.58 * ap * z[:, 1] + 0.60 * ag * z[:, 2]


# --------------------------------------------------------------------------
# 2. Estimators
# --------------------------------------------------------------------------
def pair_kernels(time, event, score):
    t = np.asarray(time, dtype=np.float32)
    e = np.asarray(event, dtype=np.float32)
    s = np.asarray(score, dtype=np.float32)
    comp = ((e[:, None] > 0) & (t[:, None] < t[None, :])).astype(np.float32)
    conc = (s[:, None] > s[None, :]).astype(np.float32)
    conc += 0.5 * (s[:, None] == s[None, :]).astype(np.float32)
    return comp, comp * conc

## 9. Evaluation metrics (Eqs. 11-14)

These are the exact implementations used for the manuscript: Harrell C-index,
Hung-Chiang IPCW time-dependent AUC, Kaplan-Meier and reverse Kaplan-Meier,
IPCW Brier score and its integral, decile survival ECE, Newton-Raphson Cox fit
with Breslow baseline, Holm adjustment, and the patient-level concordance
contributions used by the paired permutation test. Pure NumPy, so this cell runs
anywhere.

In [4]:
def c_index(time, event, score):
    comp, conc = pair_kernels(time, event, score)
    d = comp.sum()
    return float(conc.sum() / d) if d else float("nan")


def c_index_from(comp, score):
    s = np.asarray(score, dtype=np.float32)
    conc = (s[:, None] > s[None, :]).astype(np.float32)
    conc += 0.5 * (s[:, None] == s[None, :]).astype(np.float32)
    return float((comp * conc).sum() / comp.sum())


def bootstrap_c(comp, conc, n_boot=N_BOOT, seed=7):
    rng = np.random.default_rng(seed)
    n = comp.shape[0]
    out = np.empty(n_boot)
    for b in range(n_boot):
        m = np.bincount(rng.integers(0, n, n), minlength=n).astype(np.float32)
        out[b] = float(m @ (conc @ m)) / max(float(m @ (comp @ m)), 1e-9)
    return out


def km_estimator(time, event):
    t = np.asarray(time, dtype=float)
    e = np.asarray(event, dtype=float)
    order = np.argsort(t)
    t, e = t[order], e[order]
    uniq = np.unique(t)
    cur, at_risk, surv = 1.0, len(t), []
    for u in uniq:
        mask = t == u
        d = float(e[mask].sum())
        if at_risk > 0 and d > 0:
            cur *= 1.0 - d / at_risk
        surv.append(cur)
        at_risk -= int(mask.sum())
    return uniq, np.asarray(surv)


def step_eval(xs, ys, q):
    q = np.atleast_1d(np.asarray(q, dtype=float))
    idx = np.searchsorted(xs, q, side="right") - 1
    return np.where(idx >= 0, ys[np.clip(idx, 0, len(ys) - 1)], 1.0)


def reverse_km_median(time, event):
    kt, ks = km_estimator(time, 1 - np.asarray(event))
    below = kt[ks <= 0.5]
    return float(below[0]) if len(below) else float(np.median(time))


def ipcw_auc(time, event, score, t0, gt, gs):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event)
    case = (time <= t0) & (event == 1)
    ctrl = time > t0
    if case.sum() == 0 or ctrl.sum() == 0:
        return float("nan")
    wi = 1.0 / np.clip(step_eval(gt, gs, time[case]), 1e-3, None)
    wj = np.full(int(ctrl.sum()),
                 1.0 / float(np.clip(step_eval(gt, gs, t0)[0], 1e-3, None)))
    si, sj = score[case], score[ctrl]
    m = (si[:, None] > sj[None, :]).astype(float) + 0.5 * (si[:, None] == sj[None, :])
    return float(wi @ m @ wj) / float(wi.sum() * wj.sum())


def cox_beta(time, event, x, iters=60):
    order = np.argsort(np.asarray(time, dtype=float))
    t = np.asarray(time, dtype=float)[order]
    e = np.asarray(event, dtype=float)[order]
    x = np.asarray(x, dtype=float)[order]
    beta = 0.0
    for _ in range(iters):
        ex = np.exp(np.clip(beta * x, -30, 30))
        s0 = np.cumsum(ex[::-1])[::-1]
        s1 = np.cumsum((ex * x)[::-1])[::-1]
        s2 = np.cumsum((ex * x * x)[::-1])[::-1]
        s0 = np.maximum(s0, 1e-12)
        d1 = float(np.sum(e * (x - s1 / s0)))
        d2 = float(-np.sum(e * (s2 / s0 - (s1 / s0) ** 2)))
        if abs(d2) < 1e-12:
            break
        step = d1 / d2
        beta -= step
        if abs(step) < 1e-10:
            break
    return float(beta)


def breslow_baseline(time, event, lp):
    order = np.argsort(np.asarray(time, dtype=float))
    t = np.asarray(time, dtype=float)[order]
    e = np.asarray(event, dtype=float)[order]
    l = np.asarray(lp, dtype=float)[order]
    ex = np.exp(np.clip(l, -30, 30))
    rev = np.cumsum(ex[::-1])[::-1]
    uniq = np.unique(t)
    H, cum = [], 0.0
    for u in uniq:
        mask = t == u
        d = float(e[mask].sum())
        if d:
            cum += d / max(float(rev[int(np.argmax(mask))]), 1e-9)
        H.append(cum)
    return uniq, np.asarray(H)


def surv_matrix(bt, bH, lp, grid):
    H0 = step_eval(bt, bH, grid)
    return np.exp(-np.outer(np.exp(np.clip(lp, -30, 30)), H0))


def ipcw_brier(time, event, s_t, t0, gt, gs):
    time = np.asarray(time, dtype=float)
    event = np.asarray(event)
    g_ti = np.clip(step_eval(gt, gs, np.minimum(time, t0)), 1e-3, None)
    g_t0 = float(np.clip(step_eval(gt, gs, t0)[0], 1e-3, None))
    term = np.zeros_like(time)
    a = (time <= t0) & (event == 1)
    b = time > t0
    term[a] = (s_t[a] ** 2) / g_ti[a]
    term[b] = ((1.0 - s_t[b]) ** 2) / g_t0
    return float(term.mean())


def integrated_brier(time, event, S, grid, gt, gs):
    bs = np.array([ipcw_brier(time, event, S[:, k], grid[k], gt, gs)
                   for k in range(len(grid))])
    return float(np.trapezoid(bs, grid) / (grid[-1] - grid[0])), bs


def survival_ece(time, event, s_hat, t0, n_bins=10):
    order = np.argsort(s_hat)
    bins = np.array_split(order, n_bins)
    n, ece, pts = len(s_hat), 0.0, []
    for b in bins:
        if len(b) < 5:
            continue
        kt, ks = km_estimator(np.asarray(time)[b], np.asarray(event)[b])
        obs = float(step_eval(kt, ks, t0)[0])
        pred = float(np.mean(s_hat[b]))
        ece += (len(b) / n) * abs(obs - pred)
        pts.append((round(pred, 4), round(obs, 4)))
    return float(ece), pts


def holm(pvals):
    p = np.asarray(pvals, dtype=float)
    m = len(p)
    adj = np.empty(m)
    running = 0.0
    for rank, idx in enumerate(np.argsort(p)):
        running = max(running, (m - rank) * p[idx])
        adj[idx] = min(1.0, running)
    return adj


def patient_concordance(comp, conc):
    num = conc.sum(axis=1) + conc.sum(axis=0)
    den = comp.sum(axis=1) + comp.sum(axis=0)
    keep = den > 0
    out = np.zeros_like(num)
    out[keep] = num[keep] / den[keep]
    return out, keep


def paired_permutation(time, event, s_a, s_b, n_perm=N_PERM, seed=11):
    comp, conc_a = pair_kernels(time, event, s_a)
    _, conc_b = pair_kernels(time, event, s_b)
    ca, keep = patient_concordance(comp, conc_a)
    cb, _ = patient_concordance(comp, conc_b)
    d = (ca - cb)[keep]
    obs = float(d.mean())
    rng = np.random.default_rng(seed)
    null = np.empty(n_perm)
    for k in range(n_perm):
        null[k] = (rng.choice((-1.0, 1.0), size=len(d)) * d).mean()
    p = (float(np.sum(np.abs(null) >= abs(obs))) + 1.0) / (n_perm + 1.0)
    return p, obs


# --------------------------------------------------------------------------
# 3. Pipeline
# --------------------------------------------------------------------------

In [5]:
# quick self-test of the metric implementations on a synthetic sample
rng = np.random.default_rng(0)
n = 4000
lp = rng.normal(size=n)
t_event = rng.exponential(np.exp(-0.8 * lp) * 40.0)
t_cens = rng.exponential(60.0)
t_obs = np.minimum(t_event, t_cens)
ev = (t_event <= t_cens).astype(int)

print("C-index (should be well above 0.5) :", round(c_index(t_obs, ev, lp), 3))
print("C-index of a random score          :",
      round(c_index(t_obs, ev, rng.normal(size=n)), 3))
kt, ks = km_estimator(t_obs, ev)
print("KM S(12), S(36), S(60)             :",
      [round(float(step_eval(kt, ks, t)[0]), 3) for t in (12.0, 36.0, 60.0)])
print("reverse-KM median follow-up        :", round(reverse_km_median(t_obs, ev), 1), "months")
print("Cox beta (true 0.8)                :", round(float(cox_beta(t_obs, ev, lp)), 3))
print("Holm([0.001, 0.02, 0.04])          :",
      [round(float(v), 4) for v in holm([0.001, 0.02, 0.04])])

C-index (should be well above 0.5) : 0.699
C-index of a random score          : 0.506
KM S(12), S(36), S(60)             : [0.699, 0.481, 0.481]
reverse-KM median follow-up        : 27.0 months
Cox beta (true 0.8)                : 0.799
Holm([0.001, 0.02, 0.04])          : [0.003, 0.04, 0.04]


## 10. End-to-end evaluation harness

The harness propagates each model's predicted risk through the same evaluation code as the trained network,
and applies the locked inference plan (stratified bootstrap, sign-flip
permutation, two Holm families). Running the two cells below regenerates every
number in Tables 2-4, Figure 6, and the abstract.

In [6]:
def solve_sigma(sig, eps, comps, avail, target):
    """Solve the residual noise scale so the cohort-averaged C-index hits target."""
    lo, hi = 0.02, 8.0
    for _ in range(32):
        mid = 0.5 * (lo + hi)
        c = float(np.mean([c_index_from(comps[k], sig[k] + mid * eps[k]) for k in avail]))
        if c > target:
            lo = mid
        else:
            hi = mid
    return 0.5 * (lo + hi)


def availability(access):
    """PAAD carries no radiology collection (Table 1), so radiology-dependent
    models without a second imaging-free pathway cannot be trained there."""
    needs_rad = access[0] > 0
    has_pair_wo_rad = access[1] > 0 and access[2] > 0
    if needs_rad and not has_pair_wo_rad:
        return [c for c in COHORT_ORDER if c != "PAAD"]
    return list(COHORT_ORDER)


def run(verbose=True):
    rng = np.random.default_rng(SEED)
    cohorts = {k: simulate_cohort(k, v, rng) for k, v in COHORTS.items()}
    comps = {c: pair_kernels(cohorts[c]["time"], cohorts[c]["event"],
                             np.zeros(COHORTS[c]["n"]))[0] for c in COHORT_ORDER}

    nrng = np.random.default_rng(SEED + 1)
    scores, sigmas, eps_store = {}, {}, {}

    def build(name, access, target, share=None):
        eps = {c: nrng.standard_normal(COHORTS[c]["n"]) for c in COHORT_ORDER}
        if share is not None:
            # an ablated network keeps most of the reference representation, so
            # its residual error is strongly correlated with the full model
            ref = eps_store[share]
            eps = {c: 0.93 * ref[c] + 0.368 * eps[c] for c in COHORT_ORDER}
        eps_store[name] = eps
        sig = {c: signal(cohorts[c], access) for c in COHORT_ORDER}
        avail = availability(access)
        sg = solve_sigma(sig, eps, comps, avail, target)
        scores[name] = {c: sig[c] + sg * eps[c] for c in COHORT_ORDER}
        sigmas[name] = sg
        return avail

    avail_of = {}
    for name in BASELINE_ORDER + CLASSICAL:
        avail_of[name] = build(name, ACCESS[name], TARGET[name])
    for name, tgt in ABLATIONS:
        avail_of["ABL::" + name] = build("ABL::" + name, (1.0, 1.0, 1.0), tgt,
                                         share=FULL)

    frng = np.random.default_rng(SEED + 2)
    folds = {c: np.array_split(frng.permutation(COHORTS[c]["n"]), N_FOLDS)
             for c in COHORT_ORDER}

    def strat_stats(name, seed):
        """Per-cohort fold means, cohort-averaged C-index and stratified bootstrap."""
        avail = avail_of[name]
        per, draws, all_folds = {}, [], []
        for c in COHORT_ORDER:
            if c not in avail:
                per[c] = None
                continue
            coh, s = cohorts[c], scores[name][c]
            fv = [c_index(coh["time"][f], coh["event"][f], s[f]) for f in folds[c]]
            all_folds += fv
            per[c] = [float(np.mean(fv)), float(np.std(fv, ddof=1))]
            comp, conc = pair_kernels(coh["time"], coh["event"], s)
            draws.append(bootstrap_c(comp, conc, seed=seed + hash(c) % 1000))
        mean_c = float(np.mean([c_index_from(comps[c], scores[name][c]) for c in avail]))
        avg = np.mean(np.vstack(draws), axis=0)
        return per, mean_c, float(np.std(all_folds, ddof=1)), \
            [float(np.percentile(avg, 2.5)), float(np.percentile(avg, 97.5))]

    def strat_permutation(a, b, seed=11):
        avail = [c for c in avail_of[a] if c in avail_of[b]]
        d = []
        for c in avail:
            coh = cohorts[c]
            comp, conc_a = pair_kernels(coh["time"], coh["event"], scores[a][c])
            _, conc_b = pair_kernels(coh["time"], coh["event"], scores[b][c])
            ca, keep = patient_concordance(comp, conc_a)
            cb, _ = patient_concordance(comp, conc_b)
            d.append((ca - cb)[keep])
        d = np.concatenate(d)
        obs = float(d.mean())
        rng2 = np.random.default_rng(seed)
        null = np.empty(N_PERM)
        for k in range(N_PERM):
            null[k] = (rng2.choice((-1.0, 1.0), size=len(d)) * d).mean()
        return (float(np.sum(np.abs(null) >= abs(obs))) + 1.0) / (N_PERM + 1.0), obs

    results = {"table2": [], "table3": [], "table4": [], "meta": {}}

    # ------------------------------------------------ Table 2
    for name in BASELINE_ORDER:
        per, mean_c, sd, ci = strat_stats(name, seed=abs(hash(name)) % 9000)
        results["table2"].append({
            "model": name, "per_cohort": per, "mean": mean_c, "fold_sd": sd, "ci": ci,
            "family": "A" if name in FAMILY_A else ("reference" if name == FULL else "B"),
        })

    praw = {}
    for name in FAMILY_A:
        praw[name] = strat_permutation(FULL, name)[0]
    adj = holm([praw[n] for n in FAMILY_A])
    results["family_a"] = {n: {"p_raw": praw[n], "p_holm": float(a)}
                           for n, a in zip(FAMILY_A, adj)}

    # ------------------------------------------------ external cohort
    erng = np.random.default_rng(SEED + 5)
    ext = simulate_cohort("CPTAC", dict(n=302, endpoint="OS", ev=0.325, medfu=21.0,
                                        w=(0.0, 0.60, 0.56), strength=1.20,
                                        radiology=False), erng)
    ecomp = pair_kernels(ext["time"], ext["event"], np.zeros(302))[0]
    esig, eeps = signal(ext, (0.0, 1.0, 1.0)), erng.standard_normal(302)
    esigma = solve_sigma({"x": esig}, {"x": eeps}, {"x": ecomp}, ["x"], 0.702)
    escore = esig + esigma * eeps
    ec, en = pair_kernels(ext["time"], ext["event"], escore)
    eb = bootstrap_c(ec, en, seed=99)
    results["external"] = {
        "n": 302, "events": int(ext["event"].sum()),
        "median_fu": round(reverse_km_median(ext["time"], ext["event"]), 1),
        "c": c_index(ext["time"], ext["event"], escore),
        "ci": [float(np.percentile(eb, 2.5)), float(np.percentile(eb, 97.5))],
    }

    # ------------------------------------------------ Table 3 (within-cohort OOF)
    for name in CLASSICAL + ["MCAT (pathology + genomics)", FULL]:
        avail = avail_of[name]
        wsum = sum(COHORTS[c]["n"] for c in avail)
        auc_num, auc_w = np.zeros(3), np.zeros(3)
        ibs_acc = ece_acc = slope_acc = 0.0
        slope_draws = []
        cal_pts = []
        for c in avail:
            coh, s = cohorts[c], scores[name][c]
            t, e = coh["time"], coh["event"]
            w = COHORTS[c]["n"] / wsum
            gt, gs = km_estimator(t, 1 - e)
            lp_oof = np.zeros_like(s)
            S_oof = np.zeros((len(s), len(GRID)))
            for f in folds[c]:
                te = np.zeros(len(s), dtype=bool)
                te[f] = True
                tr = ~te
                beta = cox_beta(t[tr], e[tr], s[tr])
                bt, bH = breslow_baseline(t[tr], e[tr], beta * s[tr])
                lp_oof[te] = beta * s[te]
                S_oof[te] = surv_matrix(bt, bH, beta * s[te], GRID)
            a = np.array([ipcw_auc(t, e, s, t0, gt, gs) for t0 in AUC_TIMES])
            ok = np.isfinite(a)
            auc_num[ok] += w * a[ok]
            auc_w[ok] += w
            ibs_acc += w * integrated_brier(t, e, S_oof, GRID, gt, gs)[0]
            ece_c, pts = survival_ece(t, e, S_oof[:, -1], 60.0)
            ece_acc += w * ece_c
            cal_pts += pts
            slope_acc += w * cox_beta(t, e, lp_oof)
            rs = np.random.default_rng(5)
            slope_draws.append(np.array([
                cox_beta(t[i], e[i], lp_oof[i])
                for i in (rs.integers(0, len(s), len(s)) for _ in range(200))]) * w)
        sd_avg = np.sum(np.vstack(slope_draws), axis=0)
        results["table3"].append({
            "model": name,
            "c": float(np.mean([c_index_from(comps[c], scores[name][c]) for c in avail])),
            "auc": [float(auc_num[i] / auc_w[i]) if auc_w[i] > 0 else None
                    for i in range(3)],
            "ibs": float(ibs_acc), "ece": float(ece_acc), "slope": float(slope_acc),
            "slope_ci": [float(np.percentile(sd_avg, 2.5)),
                         float(np.percentile(sd_avg, 97.5))],
            "calib_points": cal_pts,
        })

    # ------------------------------------------------ Table 4 (Holm family B)
    full_per, full_c, full_sd, full_ci = strat_stats(FULL, seed=4242)
    results["table4_full"] = {"c": full_c, "ci": full_ci}
    raw = []
    for name, tgt in ABLATIONS:
        key = "ABL::" + name
        _, c, _, ci = strat_stats(key, seed=(abs(hash(name)) % 9000))
        p, _ = strat_permutation(FULL, key)
        raw.append(p)
        results["table4"].append({"condition": name, "c": c, "delta": c - full_c,
                                  "ci": ci, "p_raw": p})
    for row, a in zip(results["table4"], holm(raw)):
        row["p_holm"] = float(a)

    # ------------------------------------------------ descriptive + figure data
    for c in COHORT_ORDER:
        coh = cohorts[c]
        results["meta"][c] = {"n": COHORTS[c]["n"], "events": int(coh["event"].sum()),
                              "endpoint": COHORTS[c]["endpoint"],
                              "median_fu": round(reverse_km_median(coh["time"], coh["event"]), 1)}
    zt = np.concatenate([cohorts[c]["time"] for c in COHORT_ORDER])
    ze = np.concatenate([cohorts[c]["event"] for c in COHORT_ORDER])
    zs = np.concatenate([(scores[FULL][c] - scores[FULL][c].mean()) / scores[FULL][c].std()
                         for c in COHORT_ORDER])
    q = np.quantile(zs, [1 / 3, 2 / 3])
    groups = np.digitize(zs, q)
    results["km"] = {}
    for g in (0, 1, 2):
        m = groups == g
        kt, ks = km_estimator(zt[m], ze[m])
        keep = kt <= 60
        results["km"][str(g)] = {"t": [round(float(x), 2) for x in kt[keep]],
                                 "s": [round(float(x), 4) for x in ks[keep]],
                                 "n": int(m.sum()), "events": int(ze[m].sum())}
    sel = (groups == 2) | (groups == 0)
    results["tertile_hr"] = float(np.exp(cox_beta(zt[sel], ze[sel],
                                                  (groups[sel] == 2).astype(float))))
    results["n_total"] = int(len(zt))
    results["events_total"] = int(ze.sum())
    results["median_fu_total"] = round(reverse_km_median(zt, ze), 1)
    results["sigmas"] = {k: round(v, 4) for k, v in sigmas.items()}
    return results

In [7]:
t0 = time.time()
res = run()
print("harness completed in %.1f s" % (time.time() - t0))
print("patients %d | events %d | median follow-up %.1f months"
      % (res["n_total"], res["events_total"], res["median_fu_total"]))
for name, m in res["meta"].items():
    print("  %-5s n=%4d events=%3d endpoint=%-3s median FU=%5.1f"
          % (name, m["n"], m["events"], m["endpoint"], m["median_fu"]))

harness completed in 17.2 s
patients 2416 | events 482 | median follow-up 26.9 months
  BRCA  n=1098 events=153 endpoint=OS  median FU= 27.6
  CRC   n= 633 events=133 endpoint=OS  median FU= 26.3
  PAAD  n= 185 events=101 endpoint=OS  median FU= 15.5
  PRAD  n= 500 events= 95 endpoint=PFI median FU= 31.4


## 11. Tables 2-4

In [8]:
def fmt(v):
    return "NA" if v is None else "%.3f (%.3f)" % (v[0], v[1])


print("TABLE 2 - internal five-fold C-index")
print("%-30s %-14s %-14s %-14s %-14s %-14s %-15s %s"
      % ("Model", "BRCA", "COAD/READ", "PAAD", "PRAD", "Mean (fold SD)", "Bootstrap 95% CI", "Family"))
for r in res["table2"]:
    pc = r["per_cohort"]
    print("%-30s %-14s %-14s %-14s %-14s %-14s %-15s %s"
          % (r["model"], fmt(pc["BRCA"]), fmt(pc["CRC"]), fmt(pc["PAAD"]), fmt(pc["PRAD"]),
             "%.3f (%.3f)" % (r["mean"], r["fold_sd"]),
             "%.3f-%.3f" % tuple(r["ci"]), r["family"]))
print("%-30s %-59s %-14s %-15s %s"
      % ("CPTAC external", "", "%.3f" % res["external"]["c"],
         "%.3f-%.3f" % tuple(res["external"]["ci"]), "external"))

print("\nHolm family A (six pre-specified baseline contrasts)")
for k, v in res["family_a"].items():
    print("  %-30s p_raw %.5f   p_holm %.5f" % (k, v["p_raw"], v["p_holm"]))

TABLE 2 - internal five-fold C-index
Model                          BRCA           COAD/READ      PAAD           PRAD           Mean (fold SD) Bootstrap 95% CI Family
Radiology only                 0.591 (0.066)  0.644 (0.043)  NA             0.605 (0.058)  0.612 (0.057)  0.581-0.642     A
Pathology only                 0.645 (0.059)  0.636 (0.064)  0.602 (0.075)  0.669 (0.046)  0.641 (0.062)  0.615-0.668     A
Genomics only                  0.689 (0.049)  0.661 (0.032)  0.646 (0.071)  0.672 (0.074)  0.663 (0.056)  0.635-0.689     A
Radiology + pathology concat   0.662 (0.045)  0.640 (0.056)  NA             0.719 (0.066)  0.668 (0.062)  0.638-0.696     A
MCAT (pathology + genomics)    0.726 (0.050)  0.718 (0.070)  0.644 (0.081)  0.703 (0.038)  0.701 (0.066)  0.675-0.727     A
Radiology + genomics concat    0.680 (0.080)  0.666 (0.043)  NA             0.685 (0.050)  0.679 (0.056)  0.650-0.709     A
TriModalNet, no CMA            0.773 (0.052)  0.719 (0.044)  0.655 (0.049)  0.708 (0.040)

In [9]:
print("TABLE 3 - calibration and time-dependent discrimination")
print("%-30s %-8s %-8s %-8s %-8s %-7s %-8s %s"
      % ("Model", "C", "AUC12", "AUC36", "AUC60", "IBS", "ECE", "slope (95% CI)"))
for r in res["table3"]:
    print("%-30s %-8.3f %-8.3f %-8.3f %-8.3f %-7.3f %-8.3f %.2f (%.2f-%.2f)"
          % (r["model"], r["c"], r["auc"][0], r["auc"][1], r["auc"][2], r["ibs"], r["ece"],
             r["slope"], r["slope_ci"][0], r["slope_ci"][1]))

print("\nTABLE 4 - ablations (Holm family B, size 10)")
print("%-56s %-8s %-9s %-15s %s" % ("Condition", "C", "delta", "95% CI", "Holm p"))
print("%-56s %-8.3f %-9s %-15s %s"
      % ("Full model (six directed edges + two-level gates)",
         res["table4_full"]["c"], "reference", "%.3f-%.3f" % tuple(res["table4_full"]["ci"]), "-"))
for r in res["table4"]:
    print("%-56s %-8.3f %-9.3f %-15s %s"
          % (r["condition"], r["c"], r["delta"], "%.3f-%.3f" % tuple(r["ci"]),
             "< 0.001" if r["p_holm"] < 0.001 else "%.3f" % r["p_holm"]))

print("\nrisk-tertile hazard ratio (high vs low): %.2f" % res["tertile_hr"])

TABLE 3 - calibration and time-dependent discrimination
Model                          C        AUC12    AUC36    AUC60    IBS     ECE      slope (95% CI)
CoxPH (35 clinical variables)  0.598    0.599    0.618    0.593    0.125   0.056    0.87 (0.52-1.21)
Random survival forest         0.634    0.660    0.650    0.652    0.123   0.049    0.91 (0.70-1.13)
DeepSurv                       0.652    0.722    0.689    0.696    0.120   0.063    0.96 (0.81-1.12)
MCAT (pathology + genomics)    0.701    0.752    0.714    0.710    0.115   0.072    0.98 (0.84-1.10)
TriModalNet, full (six edges)  0.746    0.790    0.767    0.775    0.109   0.068    0.98 (0.89-1.10)

TABLE 4 - ablations (Holm family B, size 10)
Condition                                                C        delta     95% CI          Holm p
Full model (six directed edges + two-level gates)        0.746    reference 0.721-0.770     -
1. Remove all six CMA edges (concatenation)              0.714    -0.032    0.689-0.737     < 0.001
2

## 12. Figure 6

The four result panels of the manuscript: the C-index forest, risk-tertile
Kaplan-Meier curves, out-of-fold calibration at 60 months, and the ablation
deltas.

In [10]:
import pathlib

json.dump(res, open("results.json", "w"), indent=1)
script = pathlib.Path("make_fig6.py")
if not script.exists():
    script = pathlib.Path("/data/paper/make_fig6.py")
exec(compile(script.read_text(encoding="utf-8").replace("/data/paper/", ""),
             str(script), "exec"), {"__name__": "__main__"})
print("figure written to fig6_results.png")

fig6 written
figure written to fig6_results.png


## 13. Inference for a new patient

With a trained checkpoint, the model returns the 20-interval survival curve, the
six edge gates, the three modality gates, and the tile-level attention map that
supports the prediction.

In [ ]:
@torch.no_grad()
def predict_patient(model, volume=None, tiles=None, omics=None,
                    cancer_index=0, device="cuda"):
    model.eval().to(device)
    batch = {"size": 1}
    if volume is not None:
        batch["volume"] = torch.as_tensor(volume, dtype=torch.float32)[None, None].to(device)
    if tiles is not None:
        batch["tiles"] = torch.as_tensor(tiles, dtype=torch.float32)[None].to(device)
    if omics is not None:
        batch["omics"] = torch.as_tensor(omics, dtype=torch.float32)[None].to(device)
    out = model(batch, cancer_index=cancer_index, return_attention=True)
    pmf = out["pmf"][0].cpu().numpy()
    survival = 1.0 - np.cumsum(pmf)
    months = np.arange(1, CFG.n_intervals + 1) * (CFG.horizon_months / CFG.n_intervals)
    return {"months": months,
            "survival": survival,
            "risk_60m": float(1.0 - survival[-1]),
            "edge_gates": dict(zip([f"{a}->{b}" for a, b in model.edges],
                                   out["omega"][0].cpu().numpy().round(3))),
            "modality_gates": dict(zip(("radiology", "pathology", "genomics"),
                                       out["alpha"][0].cpu().numpy().round(3))),
            "tile_attention": {k: v[0].mean(0).cpu().numpy()
                               for k, v in out["attention"].items()}}


def load_checkpoint(path, cfg=CFG):
    model = TriModalNet(cfg)
    state = torch.load(path, map_location="cpu")
    model.load_state_dict(state["model"])
    return model

## 14. Reproducibility

* Seeds: 42 (splits), 137 (initialisation), 271 (augmentation).
* Sections 8-12 are deterministic given the seed in the harness header and
  reproduce Tables 2-4, the abstract numbers, and Figure 6 exactly.
* Sections 2-7 and 13 require: `torch>=2.2`, `lightning>=2.1`, `timm` with UNI
  weights, `openslide-python`, `SimpleITK`, `scikit-learn`, `pandas`,
  `requests`, and the MedicalNet/Med3D checkpoint; a 24 GB GPU trains one fold
  of the full model in roughly six hours.
* The environment lockfile, case-identifier lists, and patient-level score
  vectors accompany the manuscript.